# Estimation of Probabilistic Semi-Markovian Graphs

### *DGP (Data Generation Process)*

In [ ]:
import warnings
warnings.filterwarnings("ignore", module="dowhy")   
warnings.filterwarnings("ignore", module="tqdm")
warnings.filterwarnings("ignore", category=FutureWarning)

In [ ]:
#imports + constants
#dgp
import numpy as np
import pandas as pd
#alg1
import networkx as nx
from itertools import product, islice

import ot


q_s = [0.2, 0.5, 0.7, 0.9]
p_s = [0.2, 0.5, 0.7, 0.9]
number_of_obs = 4 # # of observed nodes
obs_nodes = [ "X", "W", "V", "Z"]#[  "W", "V","X", "Z"]   # try different number of nodes >4 7/8 fixed q p x 
seed = 40
np.random.seed(seed)
N = 15000 # change 5 - 15 for small graphs , the larger graph the more samples 
y = np.array([0.0, 1.0])
M = ot.dist(y.reshape(-1, 1), y.reshape(-1, 1), metric='sqeuclidean')# sqeuclidean -> squared wasserstein2 distance  
M /= M.max()

In [ ]:
def create_dataset(map_adj, N, number_of_obs, obs_nodes):
    map_dataset = {}
    for q in q_s:
        for p in p_s:
            nodes = []
            size = len(map_adj[(q,p)]["adj full matrix"][0])
            full_data = np.empty((size, N))
            for i in range(size):
                parents = map_adj[(q,p)]["adj full matrix"][i, :i].astype(bool)
                if not np.any(parents):
                    full_data[i] = np.random.binomial(1, 0.1, size = N) # different "q"
                else:
                    noise = np.random.binomial(1,0.1, size = N).astype(bool)
                    rows_to_consider = full_data[:i][parents].astype(bool)# takes only the rows of the beteiligte parents -> multiplication = AND
                    generated_row_wo_noise = np.bitwise_xor.reduce(rows_to_consider, axis = 0)
                    full_data[i] = np.bitwise_xor(generated_row_wo_noise, noise).astype(int)
            
            full_data = full_data.T
            for i in range(size-number_of_obs):
                nodes.append(f"U{i}")
            nodes.extend(obs_nodes)
                
            full_dataframe = pd.DataFrame(full_data, columns= nodes)
            observed_dataframe = full_dataframe.drop(columns=[col for col in full_dataframe if "U" in col], inplace=False)
            map_dataset[(q,p)] = {"full dataset": full_dataframe, "observed only":observed_dataframe}
    return map_dataset

Binary Model 

In [ ]:
# same code as for linear just w/o coefficient matrix
map_adj_1 = {}

for q in q_s:
    for p in p_s:
        m = 0 # number of uncertain edges 

        #directed edges
        probs_for_bernoulli = np.tril(np.random.rand(number_of_obs, number_of_obs), k= -1) # just random floats under hauptdiagonale
        adj_directed = (probs_for_bernoulli >0) & (probs_for_bernoulli < q) # condition for existence of directed edge
        adj_directed = adj_directed.astype(int) # converting from T/F to 1/0
        probs = np.tril(np.random.rand(number_of_obs, number_of_obs), k= -1)
        probs = probs*adj_directed
        m+= len(np.argwhere((probs>0) & (probs<1)))
        
        #bidirected edges
        bidirected_edges = {}
        for i in range(number_of_obs):
            for j in range(number_of_obs):
                if i==j: 
                    continue
                    
                bi_prob = np.random.rand()
               
                if bi_prob>0 and bi_prob < p and (i, j) not in bidirected_edges and (j, i) not in bidirected_edges:
                    edge_prob = np.random.rand()
                    bidirected_edges[(i, j)] = edge_prob
                   
        n = len(bidirected_edges)#number of unobserved nodes
        m+= n # fix to be len(np.argwhere((bidirected_edges>0) & (bidirected_edges<1)))
        adj_full = np.zeros((number_of_obs+ n,number_of_obs+ n)) # initializing with appropriate size , matrix of 1/0
        prob_full = np.zeros((number_of_obs+ n,number_of_obs+ n)) # matrix of probs of edges
        #filling the bootom right block then take first bi_edge , take first column of full matrix, take row with index len(bidirected_edges)+i then len(bidirected_edges)+j and fill out 
        adj_full[n:, n:] = adj_directed
        prob_full[n:, n:] = probs
        for idx, (i,j) in enumerate(bidirected_edges.keys()):
            
            adj_full[n+i][idx] = 1 # skip all unobserved nodes (n+) , idx regulates move to the next unobserved
            adj_full[n+j][idx] = 1

            prob_full[n+i][idx]=bidirected_edges[(i,j)]
            prob_full[n+j][idx]=bidirected_edges[(i,j)]
            
        map_adj_1[(q,p)] = {"adj full matrix":adj_full, "probability matrix":prob_full, "number of uncertain edges": m, "number of unobserved":n}


map_dataset_1 = create_dataset(map_adj_1, N, number_of_obs, obs_nodes)


## Algorithm 1

### *Splitting Graph into Subgraphs*

In [ ]:
import numpy as np
from scipy.stats import entropy
from sklearn.tree import DecisionTreeClassifier

def weights(targetColumn, parentsColumns):
    S  = np.asarray(targetColumn).ravel().astype(int)
    Pa = np.asarray(parentsColumns)

    if Pa.size == 0:                       
        noise = S#XOR with 0
    else:
        if Pa.ndim == 1:
            Pa = Pa.reshape(-1, 1)
        Pa = Pa.astype(int)
        model = DecisionTreeClassifier(min_samples_leaf=25, random_state=0)
        model.fit(Pa, S)
        pred  = model.predict(Pa).astype(int)
        noise = S ^ pred

    counts = np.bincount(noise, minlength=2)# array([2, 3])  -> two 0s, three 1s
    return float(entropy(counts)) #[2, 3] -> [0.4, 0.6] -> then −Σ p ln p 

In [ ]:
#Binary Model same as linear 
map_of_subgraphs_1 = {} # (q,p) -> subgraohs -> subgraph + probability of its existence

for q in q_s:
    for p in p_s:
        uncertain_edges = np.argwhere((map_adj_1[(q,p)]["probability matrix"]>0) & (map_adj_1[(q,p)]["probability matrix"]<1))
        sorted_u_e = uncertain_edges[uncertain_edges[:, 1].argsort()] # sorting by columns so that by increasing index k of uncertain edge we move to next column
        m = map_adj_1[(q,p)]["number of uncertain edges"] #
        n = map_adj_1[(q,p)]["number of unobserved"] #
        subgraphs = []
        for combination in product([True, False], repeat=m): # 2^m combinations , m is the length of each combination, each element is either t or f
            subgraoh_prob = map_adj_1[(q,p)]["probability matrix"].copy()
            subgraph_adj = map_adj_1[(q,p)]["adj full matrix"].copy()
            prob_of_subgraph= 1
            k = 0
            for i,v in enumerate(combination):
                if v==False:
                    if i < n: # unobserved column -> delete both entries
                        prob_of_subgraph *= 1-subgraoh_prob[sorted_u_e[k][0]][sorted_u_e[k][1]] # kth entry
                        subgraoh_prob[:, i] = 0
                        subgraph_adj[:,i] = 0
                        
                        k += 2 # deleted two subsequent entries
                    else:
                        prob_of_subgraph *= 1-subgraoh_prob[sorted_u_e[k][0]][sorted_u_e[k][1]]
                        subgraoh_prob[sorted_u_e[k][0]][sorted_u_e[k][1]] = 0
                        subgraph_adj[sorted_u_e[k][0]][sorted_u_e[k][1]] = 0
                        k+=1
                        
                    
                else:
                    prob_of_subgraph *= subgraoh_prob[sorted_u_e[k][0]][sorted_u_e[k][1]]
                    if i < n:
                        k+=2
                    else: 
                        k+=1
            mydataset = map_dataset_1[(q,p)]["observed only"]
            mynodes = list(map_dataset_1[(q, p)]["full dataset"].columns)
            Z = mynodes.index("Z")
            myparents = subgraph_adj[Z].astype(bool)
            parent_names = [mynodes[j] for j in np.where(myparents)[0]]
            observed_parents = [p for p in parent_names if p in mydataset.columns]
            Pa = mydataset[observed_parents].to_numpy()
            H = weights(mydataset["Z"].to_numpy(), Pa)
            weight = 1/ np.exp(H)
            
            subgraphs.append({"combination":combination,"adj submatrix":subgraph_adj,"prob submatrix":subgraoh_prob, "prob of subgraph": prob_of_subgraph, "weight": weight})
        map_of_subgraphs_1[(q,p)] = {"subgraphs":subgraphs}
    
                    


## Algorithm 3 

### *Estimating Causal Effect of Subgraphs*

In [ ]:
#Binary Model 1
import statsmodels.api as sm
from collections import defaultdict
import networkx as nx
from dowhy import CausalModel
from collections import defaultdict
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder

x_s_binary = [0, 1]  

map_of_estimates_and_weights_1 = {}

for q in q_s:
    for p in p_s:
        dataset = map_dataset_1[(q,p)]["observed only"]
        estimates_and_weights = {"estimate" : [], "weight": []}
        mean_z = dataset["Z"].mean()
        mean_x = dataset["X"].mean()

        for subgraph in map_of_subgraphs_1[(q,p)]["subgraphs"]:
            graph = nx.from_numpy_array(
                subgraph["adj submatrix"].T,
                create_using=nx.DiGraph)
            graph = nx.relabel_nodes(graph, dict(enumerate(
                map_dataset_1[(q,p)]["full dataset"].columns.tolist())))

            model = CausalModel(
                data=dataset,
                treatment="X",
                outcome="Z",
                graph=graph)

            try:
                identifiability = model.identify_effect(
                    proceed_when_unidentifiable=False)
                estimands = identifiability.estimands

                effect = None
                method_used = None

                # LPM via OLS
                if estimands.get("backdoor") is not None:
                    est = model.estimate_effect(
                        identifiability,
                        method_name="backdoor.linear_regression",
                        treatment_value=1,
                        control_value=0)
                    effect = est.value
                    method_used = f"backdoor{identifiability.backdoor_variables}"
                    

                elif estimands.get("frontdoor") is not None:
                    est = model.estimate_effect(
                        identifiability,
                        method_name="frontdoor.two_stage_regression",
                        treatment_value=1,
                        control_value=0)
                    effect = est.value
                    method_used = f"frontdoor{identifiability.frontdoor_variables}"
                    

                elif estimands.get("iv") is not None:
                    est = model.estimate_effect(
                        identifiability,
                        method_name="iv.instrumental_variable",
                        treatment_value=1,
                        control_value=0)
                    effect = est.value
                    method_used = f"iv{identifiability.instrumental_variables}"
                    

                if effect is None or np.isnan(effect):
                    continue

                #print(f"q={q}, p={p} {method_used} ATE={effect:.4f}")

                
                # P(Z=1 | do(X=1))
                p_z_do_x = (effect+1)/2.0
                
                p_z_do_x = float(np.clip(p_z_do_x, 0.0, 1.0))
                
                
                estimates_and_weights["estimate"].append(p_z_do_x)
                estimates_and_weights["weight"].append(subgraph["weight"])

            except Exception as e:
                print(f"Skipping subgraph q={q}, p={p}: {e}")
                continue

        map_of_estimates_and_weights_1[(q,p)] = estimates_and_weights
        

    

### *Finding Wasserstein Barycenters of Estimates*

In [ ]:
import ot # another library for Wasserstein Barycenters
import numpy as np

def barycenters(map_of_estimates_and_weights):
    
    regularization = 3e-3# regulates smoothines, smaller -> more exact

    n = 2 # number of bins, since z is binary
    y = np.array([0.0, 1.0])
    
    
    M = ot.dist(y.reshape(-1, 1), y.reshape(-1, 1), metric='sqeuclidean')# sqeuclidean -> squared wasserstein2 distance  
    M /= M.max()
    
    map_of_barycenters = {}
    
    for q in q_s:
        for p in p_s:
            
            distributions = []#among all subgraphs of a particular pair of q and p
            if map_of_estimates_and_weights[(q,p)]["estimate"] == []:
                map_of_barycenters[(q,p)] =[]
                continue
            for e in map_of_estimates_and_weights[(q,p)]["estimate"]: # e is an estimate with corresponding weight#
                probability = e
                distribution = np.array([1-probability, probability])
                distributions.append(distribution)
            A = np.stack(distributions).T 
            weights = np.array([e for e in map_of_estimates_and_weights[(q,p)]["weight"]])
            weights /= weights.sum()
            barycenter = ot.bregman.barycenter(A,M, regularization, weights)# check only for gaussian
            barycenter /= barycenter.sum()
            map_of_barycenters[(q,p)] = {"barycenter":barycenter, "A": A}
    return map_of_barycenters
    
 
map_of_barycenters_1 = barycenters(map_of_estimates_and_weights_1)
        


In [ ]:
     
        
import matplotlib.pyplot as plt
import numpy as np

qp_pairs = [(q, p) for q in q_s for p in p_s]

num_cols = len(qp_pairs)
num_rows = 2  # Row 0: All subgraph distributions (A), Row 1: Barycenter

fig, axes = plt.subplots(
    nrows=num_rows, 
    ncols=num_cols, 
    figsize=(num_cols * 5, 8), # Fixed height since we only have 2 rows
    tight_layout=True
)


axes = np.array(axes).reshape(num_rows, num_cols)


y_bins = [0.0, 1.0]

for j, (q, p) in enumerate(qp_pairs):
    
    ax1 = axes[0, j] # Top Row
    ax2 = axes[1, j] # Bottom Row
    
    
    bary_data = map_of_barycenters_1.get((q, p))
    if not bary_data or len(bary_data) == 0:
        ax1.axis('off')
        ax2.axis('off')
        ax1.text(0.5, 0.5, f"Unidentifiable\nq={q}, p={p}", fontsize=16, ha='center', va='center')
        continue

    A_matrix = bary_data["A"]
    barycenter = bary_data["barycenter"]
    
    # ---------------------------------------------------------
    # ROW 1: individual subgraph distributions
    # ---------------------------------------------------------
    # overlapping transparent bins 
    num_subgraphs = A_matrix.shape[1]
    
    # Dynamically calculate transparency based on how many subgraphs exist.
    dynamic_alpha = max(0.02, min(0.3, 3.0 / num_subgraphs))
    
    # Plot each subgraph's estimate as a highly transparent overlapping bar
    for col in range(num_subgraphs):
        ax1.bar(y_bins, A_matrix[:, col], color='blue', alpha=dynamic_alpha, width=0.4, edgecolor='none')
    
    ax1.set_title(f"Distributions q={q}, p={p}")
    ax1.set_xticks(y_bins)
    ax1.set_xlim(-0.2, 1.2) 
    ax1.set_ylim(-0.05, 1.05)
    ax1.set_ylabel("Probability")
    ax1.set_xlabel("Outcome (Z)")
    
    # ---------------------------------------------------------
    # ROW 2: Plot the final Wasserstein Barycenter
    # ---------------------------------------------------------
    # Discrete PMFs
    ax2.bar(y_bins, barycenter, color='orange', width=0.4)
    ax2.set_title(f"Barycenter q={q}, p={p}")
    ax2.set_xticks(y_bins)
    ax2.set_xlim(-0.2, 1.2)

    
    ax2.set_ylim(-0.05, 1.05)
    ax2.set_ylabel("Probability")
    ax2.set_xlabel("Outcome (Z)")

plt.show()

In [ ]:
map_of_errors_1 = {}
map_of_target_1 = {}
map_of_target_distr_1 = {}
for q in q_s:
    for p in p_s:
        data = map_dataset_1[(q,p)]["full dataset"]
        mean_z = data["Z"].mean()
        mean_x = data["X"].mean()
        estimates = []
    
        graph = nx.from_numpy_array(map_adj_1[(q,p)]["adj full matrix"].T, create_using=nx.DiGraph)
        graph = nx.relabel_nodes(graph, dict(enumerate(map_dataset_1[(q,p)]["full dataset"].columns.tolist())))
        
        
        model = CausalModel(data=data,treatment="X",outcome="Z",graph=graph)
        
        try:
            
            identifiability = model.identify_effect(
                proceed_when_unidentifiable=False)
            estimands = identifiability.estimands

            effect = None
            method_used = None

            # LPM via OLS
            if estimands.get("backdoor") is not None:
                est = model.estimate_effect(
                    identifiability,
                    method_name="backdoor.linear_regression",
                    treatment_value=1,
                    control_value=0)
                effect = est.value
                method_used = f"backdoor{identifiability.backdoor_variables}"

            elif estimands.get("frontdoor") is not None:
                est = model.estimate_effect(
                    identifiability,
                    method_name="frontdoor.two_stage_regression",
                    treatment_value=1,
                    control_value=0)
                effect = est.value
                method_used = f"frontdoor{identifiability.frontdoor_variables}"

            elif estimands.get("iv") is not None:
                est = model.estimate_effect(
                    identifiability,
                    method_name="iv.instrumental_variable",
                    treatment_value=1,
                    control_value=0)
                effect = est.value
                method_used = f"iv{identifiability.instrumental_variables}"

            if effect is None or np.isnan(effect):
                continue
            estimate = (effect+1)/2
                   
                    
                        
        except Exception as e:
            #print(f"Skip subgraph q={q}, p={p}")
            continue
        
        map_of_target_1[(q,p)] = estimate


        
            
for q in q_s:
    for p in p_s:
            
        if (q,p) not in map_of_target_1.keys():
            continue
        true_effect = map_of_target_1[(q,p)]
        distribution_te = np.array([1-true_effect, true_effect])
        #distribution_te /= distribution_te.sum() # otherwise size mismatch in emd2
        w2_error_2 = ot.emd2(distribution_te, map_of_barycenters_1[(q,p)]["barycenter"], M)
        w2_error = np.sqrt(w2_error_2)
        map_of_errors_1[(q,p)] = w2_error
        map_of_target_distr_1[(q,p)] = distribution_te


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

qp_pairs = [(q, p) for q in q_s for p in p_s]


num_rows = len(qp_pairs)
num_cols = 1  

fig, axes = plt.subplots(
    nrows=num_rows, 
    ncols=num_cols, 
    figsize=(5, num_rows * 4), # Full horizontal width, dynamic vertical stack height
    tight_layout=True
)


if num_rows == 1:
    axes = np.array([axes])
else:
    axes = np.array(axes)

y_bins = np.array([0.0, 1.0])

for j, (q, p) in enumerate(qp_pairs):
    ax = axes[j] 
    
    x_val = 0 
    bary_data = map_of_barycenters_1.get((q, p))
    true_dist = map_of_target_distr_1.get((q, p))
    
    if not bary_data or true_dist is None or len(bary_data) == 0:
        ax.axis('off')
        ax.text(0.5, 0.5, f"Unidentifiable\nq={q}, p={p}", fontsize=14, ha='center', va='center')
        continue

    A_matrix = bary_data["A"]
    barycenter = bary_data["barycenter"]
    num_subgraphs = A_matrix.shape[1]
    
 
    subgraph_cluster_width = 0.4  # Slightly wider cluster area
    slot_width = subgraph_cluster_width / num_subgraphs 
    
    # Bars take up 60% of their slot, leaving 40% empty space between them
    single_subgraph_width = slot_width * 0.4 
    main_bar_width = 0.1  
    
    # 1. Plot Subgraphs Side-by-Side with Spacing
    for col in range(num_subgraphs):
        lbl = "Individual Subgraphs" if col == 0 else ""
        
        # Start further left, move by the full slot width each iteration
        offset = -0.35 + (col * slot_width)
        
        ax.bar(y_bins + offset, A_matrix[:, col], color='#1f77b4', alpha=0.6, 
               width=single_subgraph_width,  label=lbl)
        
    # 2. Plot Wasserstein Barycenter (Centered slightly right)
    ax.bar(y_bins + 0.15, barycenter, color='#ff7f0e', width=main_bar_width, 
           edgecolor='black', linewidth=1, label="Wasserstein Barycenter")
    
    # 3. Plot True Distribution (Further right)
    ax.bar(y_bins + 0.25, true_dist, color='#2ca02c', width=main_bar_width, 
           edgecolor='black', linewidth=1, label="True Distribution")

    ax.set_title(f"Distribution q={q}, p={p}. Error {map_of_errors_1[(q,p)]:.4f}")
    ax.set_xticks(y_bins)
    ax.set_xticklabels(["Outcome 0", "Outcome 1"], fontsize=11)
    
    ax.set_xlim(-0.5, 1.5) 
    ax.set_ylim(-0.05, 1.05)
    ax.set_ylabel("Probability", fontsize=11)
    ax.set_xlabel("Outcome Z", fontsize=11)
    ax.grid(axis='y', linestyle='--', alpha=0.3)
    
    # Place the legend in a clean, non-obstructive spot
    ax.legend(loc="upper right", fontsize=10)

plt.show()
